# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Metadata as dict for display
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each `record_set`, `field`, and `column` is referenced by its `@id` as defined in the Croissant schema. This ensures you always know exactly which data entity you are working with.

In [ ]:
# Inspect available record sets & fields by their `@id` in the Croissant schema
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    rs_id = rs.id
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '[No Name]'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '[No Description]'}")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    Field @id: {field.id}, Name: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
We'll extract all record sets and their main fields for hands-on exploration.

In [ ]:
# Create a dictionary for DataFrames from all record sets, referencing via their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for RecordSet @id: {record_set_id}")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"RecordSet @id: {record_set_id} - Data shape: {dataframes[record_set_id].shape}")
    print(f"Columns: {list(dataframes[record_set_id].columns)}\n")

# For demonstration, select the first available record set with data
primary_record_set_id = next(iter(dataframes.keys())) if dataframes else None

if primary_record_set_id:
    print(f"First few rows from RecordSet @id {primary_record_set_id}:")
    display(dataframes[primary_record_set_id].head())
else:
    print("No data loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We'll use a numeric field from the sample data (select one by its field `@id` as discovered above).

In [ ]:
import numpy as np

# Automatically choose a numeric field by scanning dtypes
if primary_record_set_id is not None:
    df = dataframes[primary_record_set_id]
    # Attempt to detect numeric fields
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_fields:
        # Attempt to coerce columns to numeric
        numeric_candidate_fields = []
        for col in df.columns:
            try:
                coerced = pd.to_numeric(df[col])
                if coerced.notnull().mean() > 0.5:  # Most values numeric
                    numeric_candidate_fields.append(col)
                    df[col] = coerced  # Replace with coerced version
            except:
                continue
        numeric_fields = numeric_candidate_fields
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} above mean ({threshold:.2f}): {len(filtered_df)} records")
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to pick a categorical field for grouping
        group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        group_field = group_fields[0] if group_fields else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for analysis.")
else:
    print("No data for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if primary_record_set_id is not None and 'filtered_df' in locals() and not filtered_df.empty and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field} (filtered)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 4))
        order = filtered_df[group_field].value_counts().index[:10]  # Top 10 groups
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field, order=order)
        plt.title(f'{numeric_field} by {group_field} (top 10)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- The `mlcroissant` library makes it simple to load structured, FAIR datasets and inspect their metadata, record sets, and fields by unique Croissant `@id`.
- With a simple reference to record set and field IDs, we loaded and analyzed tabular records, filtered data, normalized numeric fields, and plotted distributions.
- For deeper analysis, use the explicit `@id` values to link provenance, data dictionaries, or cross-reference record sets as defined in the Croissant schema.